In [1]:
from common import *

In [2]:
def optimize(params, param_names, x, y):
    params = dict(zip(param_names, params))
    model = ensemble.RandomForestClassifier(**params)
    kf = model_selection.StratifiedKFold(n_splits=5)
    
    accuracies = []
    for idx in kf.split(x, y):
        train_idx,  val_idx         = idx[0],       idx[1]
        train_x,    train_y         = x[train_idx], y[train_idx]
        val_x,      val_y           = x[val_idx],   y[val_idx]
        
        model.fit(train_x, train_y)
        val_preds = model.predict(val_x)
        fold_acc = metrics.accuracy_score(val_y, val_preds)
        accuracies.append(fold_acc)
    
    # -1 because we minimize this
    return -1.0 * np.mean(accuracies)

In [3]:
param_space = [
    space.Integer       (low=   3, high= 10,    name="max_depth"),
    space.Integer       (low= 100, high=600,    name="n_estimators"),
    space.Real          (low=0.01, high=  1,    name="max_features"),
    space.Categorical   (["gini", "entropy"],   name="criterion"),
]

param_names = [
    "max_depth",
    "n_estimators",
    "max_features",
    "criterion",
]

# fix all arguments except params
optimization_func = partial(
    optimize,
    param_names=param_names,
    x=X,
    y=y
)

In [4]:
result = gp_minimize(
    optimization_func,
    dimensions=param_space,
    n_calls=15,
    n_random_starts=10,
    verbose=10,
)

Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 17.3982
Function value obtained: -0.9080
Current minimum: -0.9080
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 8.5808
Function value obtained: -0.9035
Current minimum: -0.9080
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 12.0464
Function value obtained: -0.8865
Current minimum: -0.9080
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 6.0477
Function value obtained: -0.8930
Current minimum: -0.9080
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 2.7051
Function value obtained: -0.8610
Current minimum: -0.9080
Iteration No: 6 started

In [7]:
dict(zip(param_names, result.x))

{'max_depth': np.int64(9),
 'n_estimators': np.int64(568),
 'max_features': 0.7208578478413731,
 'criterion': 'entropy'}